# 01 - Retrieval metrics: run1 (hybrid) vs run2 (+ semantic ranker)

An **offline** demo: it reads the saved run files in `results/` and computes the exact retrieval
metrics with `src/metrics.py`. No Azure keys, no network, no cost.

| Metric | Question it answers |
|---|---|
| hit rate@3 | Did the top 3 contain *any* correct chunk? |
| MRR | How high is the *first* correct chunk? (rank 1 = 1.0, rank 2 = 0.5, rank 3 = 0.33) |
| full coverage | Are *all* the needed chunks in the top 3? |

Only the 8 in-scope questions are scored. An out-of-scope question has no correct chunk.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

# Works whether Jupyter was started in the repo root or in notebooks/.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.metrics import retrieval_metrics, summarize

In [ ]:
testset = json.loads((ROOT / "data/testsets/testset_v1.json").read_text(encoding="utf-8"))
runs = {
    "run1": json.loads((ROOT / "results/run1_baseline_inputs.json").read_text(encoding="utf-8")),
    "run2": json.loads((ROOT / "results/run2_semantic_inputs.json").read_text(encoding="utf-8")),
}
needed = {s["id"]: s["reference_chunk_ids"] for s in testset}
print(len(testset), "questions;", ", ".join(f"{k}: {len(v)} rows" for k, v in runs.items()))

## Per question
`needed` lists the chunk ids a good answer requires (from the frozen test set). `retrieved` is what the
search returned, in rank order. `MRR` is the reciprocal rank of the first correct chunk; `full` is 1.0
when every needed chunk was retrieved.

In [ ]:
records = []
for s in testset:
    if s["type"] == "out_of_scope":
        continue
    record = {"id": s["id"], "type": s["type"], "needed": ",".join(needed[s["id"]])}
    for name, rows in runs.items():
        row = next(r for r in rows if r["id"] == s["id"])
        hit, rr, full = retrieval_metrics(row["retrieved_ids"], needed[s["id"]])
        record[f"{name} retrieved"] = ",".join(row["retrieved_ids"])
        record[f"{name} MRR"] = round(rr, 2)
        record[f"{name} full"] = full
    records.append(record)

pd.DataFrame(records)

## Averages over the 8 in-scope questions

In [ ]:
for name, rows in runs.items():
    _, hit, mrr, full = summarize(rows, testset)
    print(f"{name}: hit@3 = {hit:.3f}   MRR = {mrr:.3f}   full coverage = {full:.3f}")

## What to notice
- **q02, q07, q10:** the semantic ranker moved the first correct chunk to rank 1 (MRR 0.33 / 0.5 / 0.5 -> 1.0).
- **q10 still misses chunk 7:** its full coverage stays 0. Re-ranking can only reorder what the search
  already found; a missing chunk needs a larger `top_k` or different chunking.
- **hit rate cannot tell the runs apart** (1.0 for both): it hides both the ranking gain and the q10 gap.
  That is why three metrics are used.

To regenerate the inputs with live Azure calls, see the README (`python -m src.pipeline`).